# Native NCLEs, age-associated structural changes, and abundance increase

This notebook contains the three logistic regression models used to test whether native non-covalent lasso entanglements (NCLEs) and age-associated structural changes (ASC) are associated with increased protein abundance with age.

## Variables

- `entangled`: 1 = protein with native NCLEs, 0 = protein without native NCLEs
- `SC`: 1 = protein exhibiting age-associated structural change, 0 = no detected age-associated structural change
- `Abd_up`: 1 = protein abundance increases with age, 0 = protein abundance does not increase
- `length_z`: standardized protein length

All models adjust for protein length.

For each model, the notebook prints the complete `statsmodels` logistic regression summary followed by a compact odds-ratio table.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

DATA_FILE = "../data/SC_ENT_ABD.xlsx"
df = pd.read_excel(DATA_FILE)

required = ["entangled", "SC", "Abd_up", "length_z"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"N proteins = {len(df):,}")
print("\nMissing values:")
print(df[required].isna().sum())


N proteins = 1,776

Missing values:
entangled    0
SC           0
Abd_up       0
length_z     0
dtype: int64


In [3]:
def format_p(p):
    return f"{p:.3e}" if p < 0.001 else f"{p:.4f}"

def model_results(model):
    ci = model.conf_int()
    out = pd.DataFrame({
        "coef": model.params,
        "OR": np.exp(model.params),
        "OR_ci_lower": np.exp(ci[0]),
        "OR_ci_upper": np.exp(ci[1]),
        "pvalue": model.pvalues
    })
    out["p_fmt"] = out["pvalue"].map(format_p)
    return out

def print_model(model):
    print(model.summary())
    print()
    out = model_results(model)
    print(out.to_string(
        float_format=lambda x: f"{x:.3f}",
        formatters={"p_fmt": lambda x: x}
    ))
    return out


## Model 1: Native NCLE and abundance increase

### Question

Do proteins with native NCLEs have higher odds of increasing in abundance with age?

### Logistic regression model

$$
\operatorname{logit}[P(Abd\_up=1)] =
\beta_0 + \beta_1 NCLE + \beta_2 Length_z
$$

The metric of interest is the `entangled` row. Its exponentiated coefficient is the adjusted OR comparing proteins with native NCLEs with proteins without native NCLEs.


In [4]:
model1 = smf.logit("Abd_up ~ entangled + length_z", data=df).fit(disp=False)
result1 = print_model(model1)


                           Logit Regression Results                           
Dep. Variable:                 Abd_up   No. Observations:                 1776
Model:                          Logit   Df Residuals:                     1773
Method:                           MLE   Df Model:                            2
Date:                Wed, 23 Sep 2026   Pseudo R-squ.:                 0.03368
Time:                        23:41:05   Log-Likelihood:                -1018.3
converged:                       True   LL-Null:                       -1053.8
Covariance Type:            nonrobust   LLR p-value:                 3.851e-16
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.3079      0.126    -10.366      0.000      -1.555      -1.061
entangled      0.4339      0.142      3.051      0.002       0.155       0.713
length_z       0.3538      0.053      6.616      0.0

## Model 2: Age-associated structural change and abundance increase

### Question

Do proteins exhibiting age-associated structural changes have higher odds of increasing in abundance with age?

### Logistic regression model

$$
\operatorname{logit}[P(Abd\_up=1)] =
\beta_0 + \beta_1 ASC + \beta_2 Length_z
$$

The metric of interest is the `SC` row. Its exponentiated coefficient is the adjusted OR comparing proteins exhibiting age-associated structural changes with proteins without detected age-associated structural changes.


In [5]:
model2 = smf.logit("Abd_up ~ SC + length_z", data=df).fit(disp=False)
result2 = print_model(model2)


                           Logit Regression Results                           
Dep. Variable:                 Abd_up   No. Observations:                 1776
Model:                          Logit   Df Residuals:                     1773
Method:                           MLE   Df Model:                            2
Date:                Wed, 23 Sep 2026   Pseudo R-squ.:                 0.03404
Time:                        23:41:05   Log-Likelihood:                -1017.9
converged:                       True   LL-Null:                       -1053.8
Covariance Type:            nonrobust   LLR p-value:                 2.655e-16
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.0660      0.063    -16.955      0.000      -1.189      -0.943
SC             0.4065      0.124      3.267      0.001       0.163       0.650
length_z       0.4040      0.052      7.724      0.0

## Model 3: Independent associations of native NCLE and age-associated structural change

### Question

Are native NCLEs and age-associated structural changes independently associated with increased protein abundance with age?

### Logistic regression model

$$
\operatorname{logit}[P(Abd\_up=1)] =
\beta_0 + \beta_1 NCLE + \beta_2 ASC + \beta_3 Length_z
$$

The `entangled` and `SC` rows are the metrics of interest. Their ORs estimate the association of each feature with abundance increase while adjusting for the other feature and protein length.


In [6]:
model3 = smf.logit("Abd_up ~ entangled + SC + length_z", data=df).fit(disp=False)
result3 = print_model(model3)


                           Logit Regression Results                           
Dep. Variable:                 Abd_up   No. Observations:                 1776
Model:                          Logit   Df Residuals:                     1772
Method:                           MLE   Df Model:                            3
Date:                Wed, 23 Sep 2026   Pseudo R-squ.:                 0.03775
Time:                        23:41:05   Log-Likelihood:                -1014.0
converged:                       True   LL-Null:                       -1053.8
Covariance Type:            nonrobust   LLR p-value:                 3.797e-17
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.3654      0.128    -10.659      0.000      -1.616      -1.114
entangled      0.3934      0.143      2.749      0.006       0.113       0.674
SC             0.3703      0.125      2.957      0.0

## Supplementary-table summary

The table below summarizes the effect estimates for the three models in the format proposed for the Supplementary Information.


In [7]:
summary_rows = [
    {
        "Model": 1,
        "Logistic regression model": "Abundance increase ~ NCLE + protein length",
        "Adjusted OR (95% CI)": f'{result1.loc["entangled","OR"]:.2f} ({result1.loc["entangled","OR_ci_lower"]:.2f}–{result1.loc["entangled","OR_ci_upper"]:.2f})',
        "P value": format_p(result1.loc["entangled","pvalue"])
    },
    {
        "Model": 2,
        "Logistic regression model": "Abundance increase ~ ASC + protein length",
        "Adjusted OR (95% CI)": f'{result2.loc["SC","OR"]:.2f} ({result2.loc["SC","OR_ci_lower"]:.2f}–{result2.loc["SC","OR_ci_upper"]:.2f})',
        "P value": format_p(result2.loc["SC","pvalue"])
    },
    {
        "Model": 3,
        "Logistic regression model": "Abundance increase ~ NCLE + ASC + protein length",
        "Adjusted OR (95% CI)": f'NCLE: {result3.loc["entangled","OR"]:.2f} ({result3.loc["entangled","OR_ci_lower"]:.2f}–{result3.loc["entangled","OR_ci_upper"]:.2f})',
        "P value": format_p(result3.loc["entangled","pvalue"])
    },
    {
        "Model": "",
        "Logistic regression model": "",
        "Adjusted OR (95% CI)": f'ASC: {result3.loc["SC","OR"]:.2f} ({result3.loc["SC","OR_ci_lower"]:.2f}–{result3.loc["SC","OR_ci_upper"]:.2f})',
        "P value": format_p(result3.loc["SC","pvalue"])
    }
]

si_table = pd.DataFrame(summary_rows)
si_table


,Model,Logistic regression model,Adjusted OR (95% CI),P value
0,1,Abundance increase ~ NCLE + protein length,1.54 (1.17–2.04),0.0023
1,2,Abundance increase ~ ASC + protein length,1.50 (1.18–1.92),0.0011
2,3,Abundance increase ~ NCLE + ASC + protein length,NCLE: 1.48 (1.12–1.96),0.0060
3,,,ASC: 1.45 (1.13–1.85),0.0031


| Native NCLE | Age-associated structural change | Number of proteins | Abundance increased | Abundance not increased | % abundance increased |
|---|---|---:|---:|---:|---:|
| No | No | 366 | 64 | 302 | 17.5% |
| No | Yes | 66 | 17 | 49 | 25.8% |
| Yes | No | 1,011 | 298 | 713 | 29.5% |
| Yes | Yes | 333 | 119 | 214 | 35.7% |
| **Total** |  | **1,776** | **498** | **1,278** | **28.0%** |

In [8]:
# ============================================================
# Descriptive prevalence of abundance increase
# by NCLE and ASC status
# ============================================================

def prevalence_summary(data, group_col, group_name):
    """
    Calculate the number and percentage of proteins with increased
    abundance (Abd_up = 1) for each level of a binary grouping variable.
    """
    summary = (
        data.groupby(group_col)["Abd_up"]
        .agg(
            Total="count",
            Increased="sum"
        )
        .reset_index()
    )

    summary["Not increased"] = summary["Total"] - summary["Increased"]
    summary["Prevalence (%)"] = 100 * summary["Increased"] / summary["Total"]

    # Make group labels easier to read
    summary[group_name] = summary[group_col].map({
        0: f"{group_name}-",
        1: f"{group_name}+"
    })

    summary = summary[
        [group_name, "Increased", "Not increased", "Total", "Prevalence (%)"]
    ]

    return summary


# -----------------------------
# NCLE status
# -----------------------------
ncle_summary = prevalence_summary(
    df,
    group_col="entangled",
    group_name="NCLE"
)

print("Abundance increase by native NCLE status")
print(ncle_summary.to_string(
    index=False,
    formatters={"Prevalence (%)": lambda x: f"{x:.1f}%"}
))


# -----------------------------
# ASC status
# -----------------------------
asc_summary = prevalence_summary(
    df,
    group_col="SC",
    group_name="ASC"
)

print("\nAbundance increase by age-associated structural change status")
print(asc_summary.to_string(
    index=False,
    formatters={"Prevalence (%)": lambda x: f"{x:.1f}%"}
))

Abundance increase by native NCLE status
 NCLE  Increased  Not increased  Total Prevalence (%)
NCLE-         81            351    432          18.8%
NCLE+        417            927   1344          31.0%

Abundance increase by age-associated structural change status
 ASC  Increased  Not increased  Total Prevalence (%)
ASC-        362           1015   1377          26.3%
ASC+        136            263    399          34.1%
